# 3. Recurrent Neural Networks and LSTMs

RNNs process sequential data by maintaining a hidden state. This notebook covers:
- **Vanilla RNN** and the vanishing gradient problem
- **LSTM** (Long Short-Term Memory) architecture
- Sequence classification on IMDB-style data
- Character-level text generation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

%matplotlib inline
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
torch.manual_seed(42)

## 3.1 Vanilla RNN

At each time step $t$:

$$h_t = \tanh(W_{hh} h_{t-1} + W_{xh} x_t + b_h)$$
$$y_t = W_{hy} h_t + b_y$$

**Problem**: gradients through long sequences either vanish ($\to 0$) or explode ($\to \infty$).

In [ ]:
# Demonstrate vanishing gradients
# Compute product of Jacobians for a simple RNN
hidden_size = 50
W = np.random.randn(hidden_size, hidden_size) * 0.5

# Simulate gradient flow over T time steps
T_values = np.arange(1, 101)
grad_norms = []
for T in T_values:
    product = np.eye(hidden_size)
    for _ in range(T):
        # Jacobian of tanh ≈ diag(1 - tanh^2(h)) * W
        diag = np.diag(np.random.uniform(0.1, 0.9, hidden_size))  # simulated
        product = diag @ W @ product
    grad_norms.append(np.linalg.norm(product))

plt.figure(figsize=(8, 4))
plt.semilogy(T_values, grad_norms)
plt.xlabel('Sequence Length T')
plt.ylabel('Gradient Norm (log scale)')
plt.title('Vanishing Gradients in Vanilla RNN')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3.2 LSTM Architecture

LSTM introduces **gates** to control information flow:

- **Forget gate**: $f_t = \sigma(W_f [h_{t-1}, x_t] + b_f)$
- **Input gate**: $i_t = \sigma(W_i [h_{t-1}, x_t] + b_i)$
- **Cell update**: $\tilde{c}_t = \tanh(W_c [h_{t-1}, x_t] + b_c)$
- **Cell state**: $c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t$
- **Output gate**: $o_t = \sigma(W_o [h_{t-1}, x_t] + b_o)$
- **Hidden state**: $h_t = o_t \odot \tanh(c_t)$

The cell state $c_t$ acts as a "highway" for gradients.

In [ ]:
# Sequence classification: predict if sum of sequence > 0
def generate_data(n_samples=2000, seq_len=20):
    X = np.random.randn(n_samples, seq_len, 1).astype(np.float32)
    y = (X.sum(axis=1).squeeze() > 0).astype(np.int64)
    return torch.tensor(X), torch.tensor(y)

X_train, y_train = generate_data(2000, 30)
X_test, y_test = generate_data(500, 30)

class LSTMClassifier(nn.Module):
    def __init__(self, input_size=1, hidden_size=32, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 2)
    
    def forward(self, x):
        # x: (batch, seq_len, input_size)
        lstm_out, (h_n, c_n) = self.lstm(x)
        # Use last hidden state
        out = self.fc(h_n[-1])
        return out

model_lstm = LSTMClassifier().to(device)
print(model_lstm)

In [ ]:
# Compare RNN vs LSTM on the same task
class RNNClassifier(nn.Module):
    def __init__(self, input_size=1, hidden_size=32, num_layers=1):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 2)
    
    def forward(self, x):
        rnn_out, h_n = self.rnn(x)
        return self.fc(h_n[-1])

def train_model(model, X_train, y_train, X_test, y_test, n_epochs=30, lr=1e-3):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    train_accs, test_accs = [], []
    
    for epoch in range(n_epochs):
        model.train()
        optimizer.zero_grad()
        out = model(X_train.to(device))
        loss = criterion(out, y_train.to(device))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        model.eval()
        with torch.no_grad():
            train_acc = (model(X_train.to(device)).argmax(1).cpu() == y_train).float().mean().item()
            test_acc = (model(X_test.to(device)).argmax(1).cpu() == y_test).float().mean().item()
        train_accs.append(train_acc)
        test_accs.append(test_acc)
    return train_accs, test_accs

model_rnn = RNNClassifier().to(device)
model_lstm2 = LSTMClassifier().to(device)

rnn_train, rnn_test = train_model(model_rnn, X_train, y_train, X_test, y_test)
lstm_train, lstm_test = train_model(model_lstm2, X_train, y_train, X_test, y_test)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(rnn_test, 'r-', label='RNN (test)')
ax.plot(lstm_test, 'b-', label='LSTM (test)')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.set_title('RNN vs LSTM: Sequence Classification')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3.3 Character-Level Text Generation

Train an LSTM to predict the next character given a context window, then sample from it to generate text.

In [ ]:
# Simple text corpus
text = "to be or not to be that is the question whether tis nobler in the mind to suffer "
text = text * 20  # Repeat for more training data

chars = sorted(set(text))
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for c, i in char_to_idx.items()}
vocab_size = len(chars)
print(f"Vocabulary: {chars} ({vocab_size} chars)")

# Create sequences
seq_len = 20
X_chars, y_chars = [], []
for i in range(len(text) - seq_len):
    X_chars.append([char_to_idx[c] for c in text[i:i+seq_len]])
    y_chars.append(char_to_idx[text[i+seq_len]])

X_chars = torch.tensor(X_chars, dtype=torch.long)
y_chars = torch.tensor(y_chars, dtype=torch.long)

class CharLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=32, hidden_size=64):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)
    
    def forward(self, x):
        emb = self.embed(x)
        out, _ = self.lstm(emb)
        return self.fc(out[:, -1, :])

char_model = CharLSTM(vocab_size).to(device)
optimizer = optim.Adam(char_model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# Train
for epoch in range(50):
    char_model.train()
    # Mini-batch
    idx = torch.randint(0, len(X_chars), (256,))
    optimizer.zero_grad()
    loss = criterion(char_model(X_chars[idx].to(device)), y_chars[idx].to(device))
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

# Generate text
def generate_text(model, seed_text, length=100, temperature=0.8):
    model.eval()
    current = [char_to_idx[c] for c in seed_text]
    result = seed_text
    with torch.no_grad():
        for _ in range(length):
            x = torch.tensor([current[-seq_len:]]).to(device)
            logits = model(x)[0] / temperature
            probs = torch.softmax(logits, dim=0)
            idx = torch.multinomial(probs, 1).item()
            current.append(idx)
            result += idx_to_char[idx]
    return result

print("\nGenerated text:")
print(generate_text(char_model, "to be or not to be "))

## Key Takeaways

- **Vanilla RNNs** suffer from vanishing gradients for long sequences
- **LSTMs** solve this with gated memory cells that allow gradients to flow over long distances
- **Gradient clipping** (`clip_grad_norm_`) prevents exploding gradients
- For text generation, **temperature** controls randomness: low = conservative, high = creative
- Modern alternatives: GRU (simpler than LSTM) and **Transformers** (next notebook)